In [ ]:
import os, glob
import numpy as np, pandas as pd, torch
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------ CONFIG -------------------------------
DIM     = 32
DRIVE   = '/'
BASE    = f'{DRIVE}/'
PCA_DIR = f'{BASE}/PCA_{DIM}'
OUT_DIR = f'{BASE}/gathered_{DIM}'
OUT_PT  = f'{OUT_DIR}/all_users_gathered{DIM}.pt'
OUT_CSV = f'{OUT_DIR}/all_users_gathered{DIM}_mask_check.csv'

MODALITIES = ['http', 'file', 'email']
USERS = []
FEATURE_DIRS = [f'{}', f'{BASE}']
KEY, LABEL = ['date', 'user'], 'label'

ERRORS, WARNS = [], []
def err(m):  ERRORS.append(m); print('[ERROR]', m)
def warn(m): WARNS.append(m);  print('[WARN ]', m)
def ok(m):   print('[OK   ]', m)
def eq(name, got, want):
    ok(f'{name} = {got}') if got == want else err(f'{name}: {got} != {want}')
def halt(ctx):
    if ERRORS:
        raise RuntimeError(f'{ctx}: ERROR {len(ERRORS)}건\n'
                           + '\n'.join('  - ' + e for e in ERRORS))
def tload(p):
    try:    return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError: return torch.load(p, map_location='cpu')

# --------------------------- 1. 경로 검증 -----------------------------
FEATURE_DIR = next((d for d in FEATURE_DIRS if os.path.isdir(d)), None)
for p, n in [(BASE, 'BASE'), (PCA_DIR, f'PCA_{DIM}'), (FEATURE_DIR, 'feature_user')]:
    if not p or not os.path.isdir(p):
        raise FileNotFoundError(f'{n} 디렉터리가 없다: {p}')
os.makedirs(OUT_DIR, exist_ok=True)
print(f'PCA_DIR={PCA_DIR}\nFEATURE_DIR={FEATURE_DIR}\nOUT_DIR={OUT_DIR}')

# ----------------------- 2. daily_features 로딩 -----------------------
feats, ref = {}, None
for u in USERS:
    p = f'{FEATURE_DIR}/{u}_daily_features.csv'
    if not os.path.isfile(p): err(f'CSV 없음: {p}'); continue
    df = pd.read_csv(p)
    if [c for c in KEY + [LABEL] if c not in df.columns]:
        err(f'{u}: 필수 컬럼 누락 -> {list(df.columns)}'); continue
    if ref is None: ref = list(df.columns)
    elif list(df.columns) != ref:
        err(f'{u}: 컬럼 구조 불일치 -> {list(df.columns)}'); continue
    if set(df['user'].astype(str)) != {u}:
        err(f'{u}: CSV 내부 user 값 불일치'); continue
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    if df['date'].duplicated().any():
        err(f'{u}: (user,date) 중복 {int(df["date"].duplicated().sum())}건'); continue
    feats[u] = df.sort_values('date').reset_index(drop=True)
halt('daily_features 로딩')

STAT_COLS = [c for c in ref if c not in KEY + [LABEL]]
all_df = pd.concat(feats.values(), ignore_index=True)
TOTAL_ROWS, TOTAL_POS = len(all_df), int(all_df[LABEL].sum())
print(f'\n통계 feature {len(STAT_COLS)}개 | rows {TOTAL_ROWS:,} | '
      f'users {all_df["user"].nunique()} | positives {TOTAL_POS:,} '
      f'({TOTAL_POS/TOTAL_ROWS*100:.2f}%) | '
      f'{all_df["date"].min()} ~ {all_df["date"].max()}')
eq('unique user count', all_df['user'].nunique(), len(USERS))
if not all_df[LABEL].isin([0, 1]).all(): err('label에 0/1 외 값이 있다')
const = [c for c in STAT_COLS if all_df[c].nunique(dropna=False) <= 1]
if const: warn(f'상수 컬럼(유지함): {const}')
halt('daily_features 검증')

# ------------------------ 3. PCA 임베딩 로딩 --------------------------
emb = {}
for m in MODALITIES:
    for u in USERS:
        c = sorted(glob.glob(f'{PCA_DIR}/{m}/*{u}*.pt'))
        if not c: err(f'{m}/{u}: pt 없음'); continue
        if len(c) > 1: warn(f'{m}/{u}: pt {len(c)}개 -> {os.path.basename(c[0])} 사용')
        o = tload(c[0])
        d = pd.to_datetime(list(o['dates'])).strftime('%Y-%m-%d')
        v = np.asarray(o['embeddings'], dtype=np.float32)
        if int(o.get('n_components', -1)) != DIM or v.shape != (len(d), DIM):
            err(f'{m}/{u}: shape {v.shape}, n_components={o.get("n_components")} (기대 {DIM})'); continue
        if o.get('user') != u or o.get('modality') != m:
            err(f'{m}/{u}: 내부 메타 불일치 -> {o.get("user")}/{o.get("modality")}'); continue
        if d.duplicated().any(): err(f'{m}/{u}: 임베딩 날짜 중복'); continue
        if not np.isfinite(v).all(): err(f'{m}/{u}: 임베딩에 NaN/Inf'); continue
        emb[(m, u)] = pd.DataFrame(v, index=d)
halt('임베딩 로딩')
eq('임베딩 파일 수', len(emb), len(MODALITIES) * len(USERS))

# ------------------------------ 4. 결합 -------------------------------
FEATURE_NAMES = (list(STAT_COLS)
                 + [f'{m}_pc{i:03d}' for m in MODALITIES for i in range(DIM)]
                 + [f'{m}_missing' for m in MODALITIES])
D = len(FEATURE_NAMES)
print(f'\n최종 차원: 통계 {len(STAT_COLS)} + 임베딩 {len(MODALITIES)}x{DIM} '
      f'+ 결측플래그 {len(MODALITIES)} = {D}')

Xs, Ms, ys, users_, dates_, uids, offsets = [], [], [], [], [], [], {}
miss = {m: 0 for m in MODALITIES}
extra = {m: 0 for m in MODALITIES}
cur = 0

for uid, u in enumerate(USERS):
    df = feats[u]; dts = df['date'].tolist(); T = len(dts)

    raw = df[STAT_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(np.float64)
    stat_mask = np.isfinite(raw)
    if not stat_mask.all():
        warn(f'{u}: 통계 feature NaN/Inf {int((~stat_mask).sum())}개 -> 0.0, mask=False')
    stat = np.nan_to_num(raw, nan=0., posinf=0., neginf=0.).astype(np.float32)

    blocks, bmasks, flags = [], [], []
    for m in MODALITIES:
        e = emb[(m, u)]
        extra[m] += len(e.index.difference(dts))
        a = e.reindex(dts)                       # CSV 날짜 기준 정렬, 없는 날은 NaN
        present = a.notna().all(axis=1).to_numpy()
        miss[m] += int((~present).sum())
        blocks.append(a.fillna(0.0).to_numpy(np.float32))
        bmasks.append(np.repeat(present[:, None], DIM, axis=1))
        flags.append((~present).astype(np.float32)[:, None])

    Xu = np.concatenate([stat] + blocks + flags, axis=1).astype(np.float32)
    # missing 플래그 자체는 항상 관측값이므로 mask=True
    Mu = np.concatenate([stat_mask] + bmasks + [np.ones((T, 1), bool)] * len(MODALITIES), axis=1)

    if Xu.shape != (T, D):      err(f'{u}: X shape {Xu.shape} != {(T, D)}'); continue
    if Mu.shape != Xu.shape:    err(f'{u}: mask shape {Mu.shape} != {Xu.shape}'); continue
    if not np.isfinite(Xu).all(): err(f'{u}: X에 NaN/Inf 잔존'); continue
    if Xu[~Mu].any():           err(f'{u}: mask=False 위치에 0이 아닌 값'); continue

    Xs.append(Xu); Ms.append(Mu); ys.append(df[LABEL].to_numpy(np.int64))
    users_ += [u] * T; dates_ += dts; uids += [uid] * T
    offsets[u] = (cur, cur + T); cur += T
halt('결합')

X  = np.concatenate(Xs).astype(np.float32)
Mk = np.concatenate(Ms).astype(bool)
y  = np.concatenate(ys).astype(np.int64)
N  = len(X)

print()
eq('row count', N, TOTAL_ROWS)
eq('positive label count', int(y.sum()), TOTAL_POS)
eq('user count', len(set(users_)), len(USERS))
eq('(user,date) 유일', len(set(zip(users_, dates_))), N)
eq('mask shape', Mk.shape, X.shape)
if X[~Mk].any(): err('mask=False 위치에 0이 아닌 값이 있다')
else: ok(f'mask=False {int((~Mk).sum()):,}개가 모두 0.0')
bad = [u for u in USERS if dates_[slice(*offsets[u])] != sorted(dates_[slice(*offsets[u])])]
eq('시간순 깨진 사용자', bad, [])

print('\nmodality별 결측 user-day:')
for m in MODALITIES:
    print(f'  {m:6s} {miss[m]:6,d} / {N:,} ({miss[m]/N*100:5.2f}%) | '
          f'CSV에 없어 제외된 임베딩 날짜 {extra[m]:,}건')
halt('결합 후 검증')

# --------------------------- 5. 텐서 저장 -----------------------------
torch.save({
    'dim': DIM,
    'X': torch.from_numpy(X),            # (N, D) float32
    'mask': torch.from_numpy(Mk),        # (N, D) bool  True=관측
    'label': torch.from_numpy(y),        # (N,)   int64
    'users': users_, 'dates': dates_,
    'user_ids': torch.tensor(uids, dtype=torch.long),
    'user_order': list(USERS), 'user_offsets': offsets,
    'feature_names': FEATURE_NAMES,
    'stat_columns': list(STAT_COLS),
    'modalities': list(MODALITIES),
    'n_stat': len(STAT_COLS), 'n_components': DIM,
    'mask_semantics': {'true': 'observed', 'false': 'missing filled with 0.0'},
    'source_feature_dir': FEATURE_DIR, 'source_pca_dir': PCA_DIR,
}, OUT_PT)
ok(f'텐서 저장 -> {OUT_PT}')

# --------------------- 6. 육안 확인용 mask CSV -------------------------
chk = pd.DataFrame({'user': users_, 'date': dates_, 'label': y})
for i, m in enumerate(MODALITIES):
    s = len(STAT_COLS) + i * DIM
    chk[f'{m}_missing'] = (~Mk[:, s:s + DIM].any(1)).astype(np.int8)
chk['masked_stat_dims']  = (~Mk[:, :len(STAT_COLS)]).sum(1).astype(np.int16)
chk['total_masked_dims'] = (~Mk).sum(1).astype(np.int16)
for i, c in enumerate(STAT_COLS):          # 실제로 결측이 있는 통계 컬럼만 추가
    if not Mk[:, i].all(): chk[f'{c}_observed'] = Mk[:, i].astype(np.int8)
chk.to_csv(OUT_CSV, index=False)
ok(f'mask CSV 저장 -> {OUT_CSV}  shape={chk.shape}')

sample = chk[chk['total_masked_dims'] > 0]
print(f'\n마스킹된 행 {len(sample):,}개 중 상위 10개:')
print(sample.head(10).to_string(index=False) if len(sample) else '  없음')

# --------------------------- 7. 재로딩 검증 ----------------------------
r = tload(OUT_PT)
eq('reload X shape', tuple(r['X'].shape), (N, D))
eq('reload mask dtype', r['mask'].dtype, torch.bool)
eq('reload positive count', int(r['label'].sum()), TOTAL_POS)
eq('reload users/dates', (r['users'], r['dates']) == (users_, dates_), True)
if bool((r['X'][~r['mask']] != 0).any()): err('reload 후 mask=False 위치에 비0 값')
else: ok('reload 후에도 mask=False 위치가 모두 0.0')
c2 = pd.read_csv(OUT_CSV)
eq('CSV row count', len(c2), N)
for m in MODALITIES:
    eq(f'CSV {m}_missing', int(c2[f'{m}_missing'].sum()), miss[m])
halt('재로딩 검증')

print(f'\nERROR {len(ERRORS)}건 / WARNING {len(WARNS)}건')
for w in WARNS: print('  -', w)
print(f'>>> 완료: X {X.shape}, 관측 {int(Mk.sum()):,} / 마스킹 {int((~Mk).sum()):,}')